In [ ]:
import os
import re
import time
import scipy
import copy
import json
import argparse
import yaml
import shutil
import csv

import h5py
import corner
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm as tqdm

from ptemcee import ptemcee
from noise import AnalyticNoise

In [ ]:
from gbgpu.gbgpu import GBGPU
from gbgpu.thirdbody import GBGPUThirdBody
from gbgpu.utils.utility import *

from lisatools.utils.constants import YRSID_SI
from lisatools.detector import EqualArmlengthOrbits

In [ ]:
orbits = EqualArmlengthOrbits(
    force_backend="cpu"
)

gb = GBGPU(
    orbits=orbits,
    force_backend="cpu",
)

In [ ]:
def create_one_waveform(f0_center, Tobs, dt, N, params):
    gb.run_wave(*params, N=N, dt=dt, T=Tobs, oversample=None)

    df = 1 / Tobs
    f0_width = N * df

    f_min = f0_center - (f0_width / 2)
    f_max = f0_center + (f0_width / 2)

    k_min = int(np.round(f_min / df))
    freqs_grid = (np.arange(N) + k_min) * df

    noise = AnalyticNoise(freqs_grid, "MRDv1")
    psd_A = noise.psd(option="A")
    psd_E = noise.psd(option="E")

    asd_A = np.sqrt(psd_A)
    asd_E = np.sqrt(psd_E)

    i_start = (gb.start_inds - k_min).astype(np.int32)

    A_whitened = np.zeros((1, N), dtype=np.complex128)
    E_whitened = np.zeros((1, N), dtype=np.complex128)

    cols = np.arange(N)  # Shape: (128,)
    
    # we compute the corresponding source indices for the 1 signals using broadcasting: (1, 128) - (1, 1) => (1, 128)
    src_cols = cols[None, :] - i_start[:, None]

    #  we create the mask: we keep only the indices that actually exist in the source (between 0 and 1279)
    valid_mask = (src_cols >= 0) & (src_cols < N)  # Shape: (1, 128) of booleans

    # we create a grid to identify which row number (which sample) each cell belongs to
    rows = np.arange(1)[:, None]
    rows_grid = np.broadcast_to(rows, (1, N)) # Shape: (1, 128)

    A_whitened[valid_mask] = gb.A[rows_grid[valid_mask], src_cols[valid_mask]]
    E_whitened[valid_mask] = gb.E[rows_grid[valid_mask], src_cols[valid_mask]]

    A_whitened  *= np.sqrt(4 * df) / asd_A
    E_whitened  *= np.sqrt(4 * df) / asd_E
    return A_whitened, E_whitened
            

def sample_noise(variance, df, num):

    n_real = np.random.normal(
        loc=0.0, scale=np.sqrt(variance) / (np.sqrt(4.0 * df)), size=(num)
    )
    n_imag = np.random.normal(
        loc=0.0, scale=np.sqrt(variance) / (np.sqrt(4.0 * df)), size=(num)
    )

    return n_real + 1j * n_imag


def lnL_AE(s1, s2, df, Sn):
    lnL = 0.0
    for tdi in ["TDIA", "TDIE"]:
        r_ = s1[tdi] - s2[tdi]
        integrand = np.real(r_ * np.conj(r_))
        # overlap = 4.0 / Sn * df * np.sum((integrand[:-1] + integrand[1:]) / 2)
        overlap = np.sum((integrand[:-1] + integrand[1:]) / 2)
        lnL += -1.0 / 2 * overlap

    return lnL


In [ ]:
with h5py.File(
    "reconstructed_waveform.h5",
    "r"
) as f:

    print(list(f.keys()))

    for key in f.keys():
        print(
            key,
            f[key].shape,
            f[key].dtype
        )

    # Parameters
    Amp = f["amp"][:]
    Beta = f["beta"][:]
    element_index = f["element_index"][:]
    F0 = f["f0"][:]
    fddot = f["fddot"][:]
    Fdot = f["fdot"][:]
    Iota = f["iota"][:]
    Lam = f["lam"][:]
    Phi0 = f["phi0"][:]
    Psi = f["psi"][:]

    # Reconstruction information
    reconstruction_quality = f["reconstruction_quality"][:]
    residual_real = f["residual_real"][:]
    residual_reconstructed = f["residual_reconstructed"][:]
    snr = f["snr"][:]
    source_number = f["source_number"][:]
    source_real = f["source_real"][:]
    source_reconstructed = f["source_reconstructed"][:]

i = 8
amp = Amp[i]
beta = Beta[i]
f0 = F0[i]
fdot = Fdot[i]
iota = Iota[i]
lam = Lam[i]
phi0 = Phi0[i]
psi = Psi[i]

N = 128

Tobs = YRSID_SI
dt = 15.0
df = 1 / Tobs
f0_center = 0.004821699107149872

params = np.array([amp, f0, fdot, 0, phi0, iota, psi, lam, beta]) 
A, E = create_one_waveform(f0_center, Tobs, dt, N, params)

reconstucted_waveform = source_reconstructed[i] # shape : (4, 128) 4 : (A real, A imag, E real, E imag)
real_waveform = source_real[i] # shape : (4, 128) 4 : (A real, A imag, E real, E imag)
noise_reconstruced_waveform = residual_reconstructed[i] # shape : (4, 128) 4 : (A real, A imag, E real, E imag)
noise_real_waveform = residual_real[i] # shape : (4, 128) 4 : (A real, A imag, E real, E imag)

complex_reconstructed_waveform_A = reconstucted_waveform[0] + reconstucted_waveform[1] * 1j
complex_reconstructed_waveform_E = reconstucted_waveform[2] + reconstucted_waveform[3] * 1j

complex_real_waveform_A = real_waveform[0] + real_waveform[1] * 1j
complex_real_waveform_E = real_waveform[2] + real_waveform[3] * 1j

complex_noise_reconstructed_waveform_A = noise_reconstruced_waveform[0] + noise_reconstruced_waveform[1] * 1j
complex_noise_reconstructed_waveform_E = noise_reconstruced_waveform[2] + noise_reconstruced_waveform[3] * 1j

complex_noise_real_waveform_A = noise_real_waveform[0] + noise_real_waveform[1] * 1j
complex_noise_real_waveform_E = noise_real_waveform[2] + noise_real_waveform[3] * 1j

target_A = complex_real_waveform_A + complex_noise_real_waveform_A
target_E = complex_real_waveform_E + complex_noise_real_waveform_E

# target_A = complex_real_waveform_A
# target_E = complex_real_waveform_E

data_tdiae = {}
data_tdiae["TDIA"] = target_A
data_tdiae["TDIE"] = target_E

print(target_A.shape, A.shape)
plt.plot(np.abs(target_A), label="target_A")
plt.plot(np.abs(A[0]), label="A")

# plt.plot(np.abs(target_A), label="target_A")


In [ ]:
Tobs = YRSID_SI
dt = 15.0
df = 1 / Tobs
f0_center = 0.004821699107149872

SNR_min = 10
SNR_max = 100



limits_max = {'amp': 1.8e-22,
 'fdot': 1.9e-15,
 'lam': 2.0 * np.pi,
 'beta': 0.15,
 'iota': 2.25,
 'f0': 0.004823727109440232}


limits_min = {'amp': 5.0e-24,
 'fdot': 9.5e-17,
 'lam': 0.0,
 'beta': -1.0,
 'iota': 0.9,
 'f0': 0.004819671104859512 }


limits_max["beta"] = np.sin(limits_max["beta"])
limits_min["beta"] = np.sin(limits_min["beta"])
limits_max["iota"], limits_min["iota"] = np.cos(limits_min["iota"]), np.cos(limits_max["iota"])


amp = Amp[i]
sbeta = np.sin(Beta[i])
f0 = F0[i]
fdot = Fdot[i]
ciota = np.cos(Iota[i])
lam = Lam[i]
phi0 = Phi0[i]
psi = Psi[i]



param_names = ["f0", "fdot", "beta", "lam", "amp", "iota"]

param_dict = {
    "f0": f0,
    "fdot": fdot,
    "beta": sbeta,
    "lam": lam,
    "amp": amp,
    "iota": ciota,
    "phi0": phi0,
    "psi": psi,
}


prior_range = np.array([[limits_min[p], limits_max[p]] for p in param_names])


ndim = len(param_names)
nwalkers = 64
ntemps = 10
Tmax = float(2e5)
niter = 10

N = 128

# temps : niter * ntemps * nwalkers * temps pour générer une onde 


In [ ]:
noise = AnalyticNoise(f0_center, "MRDv1")
Sn = noise.psd(option="A")


def lnL_gb(data_tdiae, Sn,f0,fdot, sbeta,lam, amp, ciota):

    beta = np.arcsin(sbeta)
    iota = np.arccos(ciota)
    params = np.array([amp, f0, fdot, 0, phi0, iota, psi, lam, beta])
    A, E = create_one_waveform(f0_center, Tobs, dt, N, params)

    signal_tdiae = {}
    signal_tdiae["TDIA"] = A[0]
    signal_tdiae["TDIE"] = E[0]
    
    SNR2 = np.sum(np.abs(A) ** 2, axis = 1) + np.sum(np.abs(E) ** 2, axis = 1)
    SNR = np.sqrt(SNR2)
    return lnL_AE(data_tdiae, signal_tdiae, df, Sn), SNR

In [ ]:
print("true likelihood =",lnL_gb(data_tdiae,Sn,f0,fdot,sbeta, lam,  amp, ciota))

In [ ]:
xtruth = np.array([param_dict[p] for p in param_names])

scales = np.zeros(ndim)

epsilon = 0.1

for i, p in enumerate(param_names):

    if p == "amp":
        scales[i] = epsilon * xtruth[i]          # 5% relatif

    elif p == "f0":
        scales[i] = 0.1e-6                     # perturbation absolue

    elif p == "fdot":
        scales[i] = epsilon * xtruth[i]                       # 0.1 rad

    elif p == "beta":
        scales[i] = epsilon                    # selon ton Tobs
    elif p == "lam":
        scales[i] = epsilon                   # selon ton Tobs
    elif p == "iota":
        scales[i] = epsilon                      # 0.1 rad
    elif p == "phi0":
        scales[i] = epsilon                      # 0.1 rad
    elif p == "psi":
        scales[i] = epsilon                      # 0.1 rad




xini = np.zeros((ntemps, nwalkers, ndim))

for i in range(ndim):
    xini[:, :, i] = (
        xtruth[i]
        + scales[i] * np.random.randn(ntemps, nwalkers)
    )


betas = ptemcee.sampler.make_ladder(ndim, ntemps=ntemps, Tmax=Tmax)





def logl(x):
    # Vérifie que tous les paramètres sont dans leurs bornes
    if np.any(x < prior_range[:, 0]) or np.any(x > prior_range[:, 1]):
        return -1e99
    lnL, snr = lnL_gb(data_tdiae, Sn, *x)
    if snr > SNR_max or snr < SNR_min:
        return -1e99
    else:
        return lnL



def logp(x):
    if np.any(x < prior_range[:, 0]) or np.any(x > prior_range[:, 1]):
        return -1e99
    return 0.0

sampler = ptemcee.sampler.Sampler(nwalkers, ndim, logl, logp, logl_kwargs=[], betas=betas, adaptive=True)


chain = sampler.chain(xini)

In [ ]:
_ = chain.run(niter)

In [ ]:
chain.x.shape

burn_in = 0

chain_unfold = np.transpose(chain.x[:, 0, :, :], axes=(1, 0, 2))
lnlikevals = np.transpose(chain.logl[:, 0, :], axes=(1, 0))
lnpostvals = np.transpose(chain.logP[:, 0, :], axes=(1, 0))

thin_len = 1
n_fixed = 0
n_out = nwalkers * ((niter - burn_in - 1) // thin_len + 1)
chain_processed = np.zeros((n_out, ndim + n_fixed), dtype=float)
for i in range(ndim):
    chain_processed[:, i] = np.ravel(chain_unfold[:, burn_in::thin_len, i], order="F")
lnlike_processed = np.ravel(lnlikevals[:, burn_in::thin_len], order="F")

fig, ax = plt.subplots(1, 1, figsize=[16, 4])


for i in range(lnlikevals.shape[0]):
    steps = np.arange(lnlikevals.shape[1])
    ax.plot(steps, -lnlikevals[i], label=f"Temp {i}")

ax.set_yscale("log")
ax.set_xlabel("Steps")
ax.set_ylabel(r"$-\ln \mathcal{L}$")
ax.set_title("Evolution de la log-vraisemblance négative")
plt.show()



plt.close(fig)



In [ ]:


# Création de la figure avec un subplot par paramètre
fig, axes = plt.subplots(ndim, 1, figsize=(16, 2*ndim), sharex=True)

for i in range(ndim):
    # Tracer chaque marcheur séparément
    for j in range(nwalkers):
        axes[i].plot(chain_unfold[j, burn_in:, i], lw=0.5, alpha=0.7)
    
    # Tracer la valeur vraie
    axes[i].axhline(xtruth[i], color="red", linestyle="--", lw=1.5, label="Vraie valeur")
    
    axes[i].set_ylabel(param_names[i])
    axes[i].grid(True)
    
    # Ajouter légende sur le premier subplot seulement
    if i == 0:
        axes[i].legend(loc="upper right")

axes[-1].set_xlabel("Échantillons (après burn-in)")
plt.tight_layout()
plt.show()

In [ ]:
labels = ['f0', 'fdot0', 'sbeta', 'lam', 'amp', 'ciota']



np.save("chain_processed.npy", chain_processed)

corner.corner(chain_processed, truths=xtruth, labels=labels);